# ESG Multi-Agent Project: Milestone 1 Demo

This notebook demonstrates the Milestone 1 pipeline end to end:

1. **Knowledge base**: 15 corporate ESG reports, chunked (~500 tokens, 50 overlap) and embedded with Qwen `text-embedding-v3`, stored in ChromaDB
2. **Retrieval (R)**: a natural-language query → the 5 most relevant chunks, optionally filtered by company
3. **Retrieval accuracy**: precision@5 on 10 hand-labelled test queries
4. **Generation (G)**: the Generative Agent writes a cited ESG report section using only retrieved context

```
PDFs ──► chunk_pdfs.py ──► chunks.json ──► embed_and_store.py ──► ChromaDB
                                                                    │
                       query ──► retrieve.py (embed + similarity search)
                                                                    │
                                     top-k chunks ──► generative_agent.py (Qwen) ──► cited draft
```

> Run this notebook from the project folder with the `venv` kernel, and with `DASHSCOPE_API_KEY` set in `.env`.

## 1. Setup

In [2]:
import json
import collections
import pandas as pd
from IPython.display import Markdown, display

from retrieve import retrieve, collection
from generative_agent import generate_section, PILLARS, MODEL

pd.set_option("display.max_colwidth", 120)
print("Setup OK. Chat model:", MODEL)

Setup OK. Chat model: qwen-plus


## 2. The knowledge base

How many chunks each company contributes. The imbalance (e.g. Shell and HSBC vs Microsoft) comes from report length and is worth keeping in mind when comparing companies.

In [3]:
print("Total chunks in ChromaDB:", collection.count())

metas = collection.get(include=["metadatas"])["metadatas"]
kb = (pd.DataFrame(metas)
        .groupby(["company", "industry"]).size()
        .reset_index(name="chunks")
        .sort_values("chunks", ascending=False))
kb

Total chunks in ChromaDB: 5287


,company,industry,chunks
11,Shell,Energy / Oil & Gas,1128
3,HSBC,Financial Services,1092
12,Sony,Technology / Electronics,455
4,Honda,Automotive,438
9,Nestle,Consumer Goods / Food & Beverage,396
13,Toyota Motor Corporation,Automotive,302
8,Mitsubishi Corporation,Diversified / Industrial Conglomerate,230
1,Apple,Technology / Electronics,216
10,Samsung,Technology / Electronics,190
14,Unilever,Consumer Goods,181


## 3. Retrieval demo

`retrieve(query, company, k)` embeds the query with the same model used for the chunks, then asks ChromaDB for the `k` nearest chunks. **Lower distance = more similar.**

In [4]:
def show_results(query, company=None, k=5, preview=300):
    results = retrieve(query, company=company, k=k)
    print(f"Query: {query!r}   Company filter: {company or 'none'}\n")
    for rank, r in enumerate(results, 1):
        text = " ".join(r["text"].replace("\x00", "").split())[:preview]
        print(f"#{rank}  distance={r['distance']:.3f}  [{r['metadata']['company']}]  {r['id']}")
        print(f"    {text}...\n")
    return results

_ = show_results("Scope 1 and Scope 2 greenhouse gas emissions", company="Apple")

Query: 'Scope 1 and Scope 2 greenhouse gas emissions'   Company filter: Apple

#1  distance=0.542  [Apple]  Apple_Environmental_Progress_Report_2025_chunk_166
    emissions. Location Scope 1 x Scope 2 Total gas (MMBtu) Renewable biogas (MMBtu) Scope 1 emissions (metric tons CO2e) Electricity (million kWh) Renewable electricity (million kWh) Scope 2 emissions (market-based, metric tons CO2e)1 Corporate 354,028 625,053 55,200 1,055 1,055 278,108 Cupertino, CA 2...

#2  distance=0.782  [Apple]  Apple_Environmental_Progress_Report_2025_chunk_180
    Parameter Quantity Units Boundary / Protocol Natural Gas Consumption 1,048,000 Metric million British thermal unit Worldwide occupied properties / Apple Internal Protocol Electricity Consumption 3,800 Million kilowatt hours (mkWh) Worldwide occupied properties / Apple Internal Protocol Renewable Ene...

#3  distance=0.793  [Apple]  Apple_Environmental_Progress_Report_2025_chunk_118
    3 emissions from purchased goods and services, other fuel a

### 3a. Why the company filter matters

Without a filter, the search returns the closest chunks from *any* company. With a filter, results come only from that company's report, which is what the Generative Agent needs.

In [5]:
_ = show_results("water withdrawal and water consumption", company=None, k=5, preview=150)

Query: 'water withdrawal and water consumption'   Company filter: none

#1  distance=0.543  [Coca Cola]  Sustainability-Statement_chunk_90
    at site level using the environmental management system. Total water consumed Water consumption measures water used by CCEP in our production of bever...

#2  distance=0.667  [Nestle]  non-financial-statement-2025_chunk_107
    23.9 Water consumed in regions with High or Extremely High Baseline Water Stress (percentage) 55.5 53.5 Water withdrawn in regions with High or Extrem...

#3  distance=0.683  [Honda]  honda-SR-2025-en-all_chunk_131
    water intake Waste generated Calculation method: Amount of water intake = Σ (Purchased from water utilities + Groundwater intake + Rainwater utilizati...

#4  distance=0.715  [Honda]  honda-SR-2025-en-all_chunk_130
    �ɾɾɾɾɾɾɾɾɾɾ 23 Pollution ɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾ 40 Water ɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾ 43 Biodiversity and Ecosystems ɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾɾ 45 Resource ...

#5  distance=0.734  [Lenovo]  FY2026

In [6]:
_ = show_results("water withdrawal and water consumption", company="Coca Cola", k=5, preview=150)

Query: 'water withdrawal and water consumption'   Company filter: Coca Cola

#1  distance=0.543  [Coca Cola]  Sustainability-Statement_chunk_90
    at site level using the environmental management system. Total water consumed Water consumption measures water used by CCEP in our production of bever...

#2  distance=0.745  [Coca Cola]  Sustainability-Statement_chunk_66
    disclosed at Group level only. Strategic Report Governance and Directors’ Report Financial Statements Sustainability Statement Other Information Coca-...

#3  distance=0.755  [Coca Cola]  Sustainability-Statement_chunk_91
    , not RTD litres). Non-production sites are excluded and production facilities linked to coffee roasting, PET preforms and recycling are out of scope....

#4  distance=0.791  [Coca Cola]  Sustainability-Statement_chunk_41
    in our finished drinks, at an aggregate level, to nature and communities(A) KPI: Water returned as a percentage of total sales volume through replenis...

#5  distance=0.805 

### 3b. Try your own query

Change `MY_QUERY` and `MY_COMPANY` and re-run. Valid company names are listed in the table in section 2.

In [7]:
MY_QUERY = "board independence and diversity of directors"
MY_COMPANY = "Shell"

_ = show_results(MY_QUERY, company=MY_COMPANY)

Query: 'board independence and diversity of directors'   Company filter: Shell

#1  distance=0.588  [Shell]  shell-integrated-annual-and-sustainability-report_chunk_379
    well as complementing Shell's wider diversity policies and embracing Shell's values, Code of Conduct and sustainability goals. Currently, this policy is not applied to the individual Committees, although we strive to apply diverse representation across the Committees. DE&I across Committee membershi...

#2  distance=0.705  [Shell]  shell-integrated-annual-and-sustainability-report_chunk_757
    or in part. To avoid unnecessary duplication, our disclosures are designed to address these requirements in an integrated manner where possible. Our disclosures in the Sustainability Statements do not include information stemming from other legislation that is additional to the requirements prescrib...

#3  distance=0.715  [Shell]  shell-integrated-annual-and-sustainability-report_chunk_380
    an inclusive Board environment,

## 4. Retrieval accuracy: precision@5

Each of the 10 test queries returned 5 chunks, which were labelled by hand (1 = a report writer would cite this chunk for the query, 0 = not).
Precision@5 = relevant results / 5. Produced by `eval_retrieval.py run` → manual labelling → `eval_retrieval.py score`.

In [8]:
df = pd.read_csv("results/retrieval_eval.csv", encoding="utf-8-sig")
df["relevant"] = pd.to_numeric(df["relevant"], errors="coerce")

if df["relevant"].isna().any():
    print(f"{df['relevant'].isna().sum()} rows are not labelled yet. Finish labelling, then re-run this cell.")
else:
    p5 = (df.groupby(["query_id", "company", "query"])["relevant"].sum() / 5).rename("precision@5").reset_index()
    display(p5)
    print(f"Mean precision@5: {p5['precision@5'].mean():.2f}")

1 rows are not labelled yet. Finish labelling, then re-run this cell.


## 5. Generative Agent: write one section

For a pillar, the agent runs several sub-queries (below), merges and de-duplicates the retrieved chunks, and asks Qwen to write a section **using only that context**, citing chunk IDs and listing disclosure gaps.

*This makes one LLM call and takes about 20–60 seconds.*

In [9]:
COMPANY = "Apple"
PILLAR = "Environmental"   # "Environmental", "Social" or "Governance"

print("Sub-queries used for retrieval:")
for q in PILLARS[PILLAR]["queries"]:
    print("  -", q)

Sub-queries used for retrieval:
  - Scope 1, Scope 2 and Scope 3 greenhouse gas emissions
  - energy consumption and renewable energy share
  - water withdrawal and consumption
  - waste generated, recycling and circularity
  - net-zero and climate targets


In [10]:
section, sources = generate_section(COMPANY, PILLAR)
print(f"Retrieved {len(sources)} unique chunks as context.\n")
display(Markdown(f"## {COMPANY}: {PILLAR}\n\n{section}"))

Retrieved 19 unique chunks as context.



## Apple: Environmental

### GRI 302 Energy  
Apple achieved 100% renewable electricity for its global corporate facilities—including offices, retail stores, and data centers—starting in January 2018 [Apple_Environmental_Progress_Report_2025_chunk_126]. In fiscal year 2024, Apple reported 3,777,000 MWh of renewable electricity used across corporate facilities, representing 100% of total energy use [Apple_Environmental_Progress_Report_2025_chunk_125]. This includes a diversified portfolio of on-site and off-site renewable generation: solar (72%), wind (27%), microhydro (0.2%), and biogas fuel cells (0.2%), with a total operational capacity of 1,782 MW across 22 countries and regions [Apple_Environmental_Progress_Report_2025_chunk_165]. For colocation facilities—third-party data centers where Apple leases capacity—100% of Apple’s energy use was matched with renewable energy generated within the same country or regional grid since January 2018 [Apple_Environmental_Progress_Report_2025_chunk_173]. In FY 2024, Apple’s corporate facilities consumed 1,055 million kWh of electricity, all sourced renewably [Apple_Environmental_Progress_Report_2025_chunk_166].

### GRI 303 Water  
In fiscal year 2024, Apple’s corporate facilities withdrew 1,756 million gallons of water globally, of which 1,532 million gallons were freshwater and 197 million gallons were recycled water [Apple_Environmental_Progress_Report_2025_chunk_126]. Recycled water is sourced primarily from municipal treatment plants (≥95%) and used for irrigation, cooling tower makeup, and toilet flushing [Apple_Environmental_Progress_Report_2025_chunk_128]. Apple also utilized 27 million gallons from other alternative sources—including rainwater and recovered condensate—in FY 2024 [Apple_Environmental_Progress_Report_2025_chunk_127]. The company reported 900 million gallons of water discharged globally in FY 2024 [Apple_Environmental_Progress_Report_2025_chunk_181]. Apple certifies all Apple-owned data centers to the Alliance for Water Stewardship (AWS) Standard by 2025, and as of FY 2024, 42% of supplier participants in the Supplier Clean Water program achieved an average 50% water reuse rate target [Apple_Environmental_Progress_Report_2025_chunk_4].

### GRI 305 Emissions  
Apple has been carbon neutral for corporate (Scope 1 + Scope 2) emissions since April 2020 and maintained this status through FY 2024 via renewable energy procurement, energy efficiency, and high-quality carbon credits [Apple_Environmental_Progress_Report_2025_chunk_118][Apple_Environmental_Progress_Report_2025_chunk_8]. In FY 2024, Apple’s market-based Scope 2 emissions from purchased electricity were zero; however, emissions from purchased steam, heating, and cooling totaled 3,300 metric tons CO₂e [Apple_Environmental_Progress_Report_2025_chunk_165]. Total corporate Scope 1 and Scope 2 emissions (market-based) were 278,108 metric tons CO₂e for corporate facilities and 837,859 metric tons CO₂e for data centers [Apple_Environmental_Progress_Report_2025_chunk_166]. Apple’s comprehensive carbon footprint—including Scope 3 emissions from manufacturing (29.03 million metric tonnes), product use (4.68 million metric tonnes), transportation, business travel, employee commute, and end-of-life—was 15.3 million metric tonnes CO₂e gross in FY 2024, with 0.7 million metric tonnes offset, yielding net emissions of 14.5 million metric tonnes CO₂e [Apple_Environmental_Progress_Report_2025_chunk_9]. Apple’s 2030 target is a 75% reduction in gross emissions versus the 2015 baseline (38.4 million metric tonnes) [Apple_Environmental_Progress_Report_2025_chunk_4].

### GRI 306 Waste  
Apple’s corporate facilities achieved a landfill diversion rate of 70% in FY 2024 [Apple_Environmental_Progress_Report_2025_chunk_127]. In that year, 18,800 metric tonnes of trash were disposed of in landfill, 1,200 metric tonnes of hazardous (regulated) waste were generated, and 36,500 metric tonnes were recycled [Apple_Environmental_Progress_Report_2025_chunk_181]. Additionally, 4,100 metric tonnes were composted and 1,000 metric tonnes were sent to waste-to-energy facilities [Apple_Environmental_Progress_Report_2025_chunk_181]. Apple’s goal is zero waste to landfill from corporate facilities and suppliers [Apple_Environmental_Progress_Report_2025_chunk_4]. Supply chain waste diverted from landfill totaled 600,000 metric tonnes in FY 2024 [Apple_Environmental_Progress_Report_2025_chunk_127].

### TCFD Metrics & Targets  
Apple aligns its climate targets with a 1.5°C pathway, including a commitment to carbon neutrality across its entire value chain by 2030—requiring a 75% reduction in gross emissions from the 2015 baseline [Apple_Environmental_Progress_Report_2025_chunk_4][Apple_Environmental_Progress_Report_2025_chunk_9]. The company discloses its full value chain emissions annually via CDP and its Environmental Progress Report [Apple_Environmental_Progress_Report_2025_chunk_9]. Apple reports Scope 3 emissions using the WRI/WBCSD GHG Protocol Value Chain standard, covering purchased goods and services, transportation, business travel, employee commute (including work-from-home), fuel- and energy-related activities, and product use [Apple_Environmental_Progress_Report_2025_chunk_181][Apple_Environmental_Progress_Report_2025_chunk_118]. However, no absolute or intensity-based near-term (2025–2027) science-based target aligned with SBTi criteria is disclosed in the provided context.

### Disclosure gaps  
- No quantitative data on Scope 3 emissions from upstream transportation and distribution, end-of-life processing, or investments.  
- No disclosure of water stress assessments per facility or basin-level water withdrawal data aligned with Aqueduct Water Risk Atlas.  
- No reporting on Scope 1 and Scope 2 emissions intensity (e.g., per employee or per revenue dollar) for trend analysis.  
- No verification statement or assurance level for Scope 3 emissions beyond Business Travel and Employee Commute (per Apex conclusion in chunk_183).  
- No disclosure of TCFD-aligned scenario analysis, climate risk financial impacts, or board-level oversight metrics.  
- No data on wastewater quality parameters (e.g., pH, heavy metals, BOD/COD) or discharge compliance rates.

### 5a. Grounding check

Check that every citation points to a real chunk, and that each number in the text actually appears in the chunk it cites.

In [11]:
import re

all_chunks = collection.get(ids=sources, include=["documents"])
source_text = dict(zip(all_chunks["ids"], all_chunks["documents"]))

cited = re.findall(r"\[([^\[\]]+?_chunk_\d+)\]", section)
unknown = sorted({c for c in cited if c not in source_text})
print(f"Citations: {len(cited)} total, {len(set(cited))} unique, {len(unknown)} not matching a retrieved chunk")
if unknown:
    print("  Unmatched:", unknown[:10])

found = missing = 0
for sentence, cid in re.findall(r"([^.\n]*?\[([^\[\]]+?_chunk_\d+)\])", section):
    src = source_text.get(cid, "").replace(",", "")
    for num in re.findall(r"\d[\d,.]*\d", sentence.split("[")[0]):
        if num.replace(",", "") in src:
            found += 1
        else:
            missing += 1
total = found + missing
print(f"Numbers found in their cited chunk: {found}/{total}" + (f" ({found/total:.0%})" if total else ""))

Citations: 26 total, 12 unique, 0 not matching a retrieved chunk
Numbers found in their cited chunk: 34/42 (81%)


## 6. Full draft report

`python generative_agent.py "<Company>"` runs all three pillars and saves a Markdown report with a table of contents to `results/drafts/`.

In [12]:
import glob, os

drafts = sorted(glob.glob("results/drafts/*.md"), key=os.path.getmtime)
print("Saved drafts:", [os.path.basename(d) for d in drafts])
if drafts:
    with open(drafts[-1], encoding="utf-8") as f:
        display(Markdown(f.read()[:4000] + "\n\n*... (truncated, open the file for the full report)*"))

Saved drafts: ['Apple_draft_v0.md', 'Apple_draft_v1.md']


# Apple ESG Report (Draft v1)

*Generated 2026-10-20 by Generative Agent v1 (qwen-plus)*

## Table of Contents

1. [Environmental](#environmental)
2. [Social](#social)
3. [Governance](#governance)

## Environmental

### GRI 302 Energy  
Apple achieved 100% renewable electricity for its global corporate facilities—including offices, retail stores, and data centers—starting in January 2018 [Apple_Environmental_Progress_Report_2025_chunk_126]. In fiscal year 2024, Apple reported 3,777,000 MWh of renewable electricity used across corporate facilities, representing 100% of total energy use [Apple_Environmental_Progress_Report_2025_chunk_125]. This includes on-site generation from a diversified portfolio of renewable technologies: solar (72%), wind (27%), microhydro (0.2%), and biogas fuel cells (0.2%), with a total operational capacity of 1,782 MW across 22 countries and regions [Apple_Environmental_Progress_Report_2025_chunk_165]. For colocation facilities—third-party data centers where Apple leases capacity—100% of Apple’s energy use has been matched with renewable energy generated within the same country or regional grid since January 2018 [Apple_Environmental_Progress_Report_2025_chunk_173]. In FY 2020, Apple’s colocation facilities consumed 372,901,398 kWh of electricity, all matched to renewables, resulting in zero scope 2 emissions from that segment [Apple_Environmental_Progress_Report_2025_chunk_173].

### GRI 303 Water  
In fiscal year 2024, Apple’s corporate facilities withdrew 1,756 million gallons of water globally, of which 1,532 million gallons were freshwater and 197 million gallons were recycled water [Apple_Environmental_Progress_Report_2025_chunk_126]. Recycled water is sourced primarily from municipal treatment plants (≥95%) and used for irrigation, cooling tower makeup, and toilet flushing [Apple_Environmental_Progress_Report_2025_chunk_128]. Apple also reported 27 million gallons from “other alternative sources”, including rainwater and recovered condensate [Apple_Environmental_Progress_Report_2025_chunk_127]. Total water discharge was 900 million gallons worldwide [Apple_Environmental_Progress_Report_2025_chunk_181]. Apple defines facility water use as “high stress” if located in basins classified as high or extremely high baseline water stress per WRI’s Aqueduct Water Risk Atlas V4.0 [Apple_Environmental_Progress_Report_2025_chunk_213]. The company certified all Apple-owned data centers to the Alliance for Water Stewardship (AWS) Standard by 2025 [Apple_Environmental_Progress_Report_2025_chunk_4], and engaged in collaborative initiatives—including co-developing the Volumetric Water Benefit Accounting methodology with WRI—to advance technical best practices in water stewardship [Apple_Environmental_Progress_Report_2025_chunk_74].

### GRI 305 Emissions  
Apple achieved carbon neutrality for its corporate (Scope 1 + Scope 2) emissions beginning in April 2020 and has maintained it annually through high-quality carbon credit retirements [Apple_Environmental_Progress_Report_2025_chunk_118]. In fiscal year 2024, Apple’s corporate Scope 1 emissions totaled 55,200 metric tons CO₂e, and Scope 2 (market-based) emissions totaled 278,108 metric tons CO₂e across all corporate locations—including data centers, retail stores, and offices [Apple_Environmental_Progress_Report_2025_chunk_166]. Notably, Apple’s Scope 2 emissions from purchased electricity are zero; however, emissions from purchased steam, heating, and cooling amounted to 3,300 metric tons CO₂e in FY 2024 [Apple_Environmental_Progress_Report_2025_chunk_165]. Apple’s comprehensive carbon footprint—including Scope 3 categories such as product use (4.68 million metric tonnes), manufacturing (29.03 million metric tonnes), transportation, and recycling—was verified by a non-Apex third-party provider [Apple_Environmental_Progress_Report_2025_chunk_183]. Apple’s net corporate emissions in FY 2024 were 14.5 million metric tons CO₂e (gross: 15.3 million; offsets/removal:

*... (truncated, open the file for the full report)*

## 7. Observations & next steps

- *Fill in: precision@5 result and where retrieval failed (e.g. table-heavy chunks, table-of-contents pages).*
- *Fill in: what the grounding check showed (citation accuracy, unsupported claims).*
- Apple's source is an *Environmental* Progress Report, so the Social and Governance sections are mostly disclosure gaps. The agent reports these gaps instead of inventing content.
- **Milestone 2:** an Assessment Agent will score each section against a 6-criterion rubric, and its feedback will be passed back into `generate_section(..., previous_draft, feedback)` in a LangGraph generate → assess → revise loop.